# Inverse Problem: recover the convection speed with a PINN

PDE:  $u_t + c\,u_x = 0$  with **unknown** speed $c$.

We are given only a few **sparse, noisy sensor measurements** of $u$ at scattered
points $(x_i, t_i)$. We do **not** know $c$, and we have **no** initial/boundary
condition. Goal: recover $c$ **and** the full field $u(x,t)$.

### Why this favors a PINN over classic CFD
- **CFD (FTBS)** solves the *forward* problem: you must already know $c$. To infer
  $c$ from data you wrap the solver in an outer optimization loop and run a **full
  simulation for every trial value of $c$** (and differentiating through the solver
  is painful).
- **PINN** makes $c$ a single trainable parameter and learns it **jointly** with the
  field in one training run. The PDE residual is the regularizer; the sparse data
  anchors the solution. Mesh-free, gradient-based, no forward solves in a loop.

Runs in a few seconds on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Imports, device, and the (hidden) ground truth
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

C_TRUE = 1.3          # <-- the speed we PRETEND not to know
L, T   = 2.0, 1.0     # domain [0,L] x [0,T]

def u0(x):            # smooth Gaussian initial profile
    exp = torch.exp if torch.is_tensor(x) else np.exp
    return exp(-((x - 0.4) ** 2) / (2 * 0.12 ** 2))

def exact(x, t):      # analytic solution used ONLY to fake sensor data
    return u0(x - C_TRUE * t)

In [ ]:
# Cell 2 -- Generate sparse, NOISY measurements (this is all the PINN sees)
N_DATA = 60                       # only 60 scattered sensor readings
NOISE  = 0.02                     # 2% Gaussian noise

xd = np.random.rand(N_DATA) * L
td = np.random.rand(N_DATA) * T
ud = exact(xd, td) + NOISE * np.random.randn(N_DATA)

xd = torch.tensor(xd, dtype=torch.float32, device=device).reshape(-1, 1)
td = torch.tensor(td, dtype=torch.float32, device=device).reshape(-1, 1)
ud = torch.tensor(ud, dtype=torch.float32, device=device).reshape(-1, 1)
print(f'{N_DATA} noisy measurements generated (true c = {C_TRUE}).')

In [ ]:
# Cell 3 -- PINN with c as a TRAINABLE parameter (initial guess deliberately wrong)
class InversePINN(nn.Module):
    def __init__(self, h=32, c_init=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
        # the unknown physical parameter, learned like any weight
        self.c = nn.Parameter(torch.tensor(c_init))
    def forward(self, x, t):
        return self.net(torch.cat([x, t], 1))

model = InversePINN(c_init=0.3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
mse = nn.MSELoss()

c_history = []
t0 = time.perf_counter()
for e in range(3000):
    opt.zero_grad()
    # PDE residual on random collocation points, using the CURRENT guess of c
    xi = (torch.rand(2000, 1, device=device) * L).requires_grad_(True)
    ti = (torch.rand(2000, 1, device=device) * T).requires_grad_(True)
    ui = model(xi, ti)
    u_t = torch.autograd.grad(ui, ti, torch.ones_like(ui), create_graph=True)[0]
    u_x = torch.autograd.grad(ui, xi, torch.ones_like(ui), create_graph=True)[0]
    loss_pde  = mse(u_t + model.c * u_x, torch.zeros_like(ui))
    loss_data = mse(model(xd, td), ud)          # fit the sparse measurements
    loss = loss_pde + 10.0 * loss_data
    loss.backward(); opt.step()

    c_history.append(model.c.item())
    if e % 500 == 0:
        print(f'epoch {e:4d}  loss {loss.item():.2e}  c = {model.c.item():.4f}')
if device.type == 'cuda':
    torch.cuda.synchronize()
train_time = time.perf_counter() - t0

c_est = model.c.item()
print(f'\nTraining time     : {train_time:.3f} s')
print(f'Recovered c       : {c_est:.4f}')
print(f'True c            : {C_TRUE}')
print(f'Relative error    : {abs(c_est - C_TRUE) / C_TRUE * 100:.2f} %')

In [ ]:
# Cell 4 -- Results: c converges to the truth, and the field is reconstructed
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# (left) convergence of the learned speed toward the true value
ax[0].plot(c_history, 'b-')
ax[0].axhline(C_TRUE, color='g', ls='--', label=f'true c = {C_TRUE}')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('learned c')
ax[0].set_title('Inverse recovery of convection speed'); ax[0].legend(); ax[0].grid(alpha=.3)

# (right) reconstructed field at t=T vs exact, plus the noisy data used
x = np.linspace(0, L, 200)
with torch.no_grad():
    xe = torch.tensor(x, dtype=torch.float32, device=device).reshape(-1, 1)
    u_pred = model(xe, torch.full_like(xe, T)).cpu().numpy().ravel()
ax[1].plot(x, exact(x, T), 'g', lw=2.5, label='exact @ t=T')
ax[1].plot(x, u_pred, 'r-.', label='PINN @ t=T')
mask = (td.cpu().numpy().ravel() > 0.8)          # show data near t=T
ax[1].scatter(xd.cpu().numpy().ravel()[mask], ud.cpu().numpy().ravel()[mask],
              c='k', s=20, label='noisy data (t>0.8)')
ax[1].set_xlabel('x'); ax[1].set_ylabel('u')
ax[1].set_title('Reconstructed field'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Takeaways
- The PINN recovered the unknown speed $c$ to within ~1% from just 60 noisy points,
  **while simultaneously reconstructing the full field** $u(x,t)$ — in one training run.
- Doing the same with FTBS would mean an outer loop calling a full forward solve for
  every candidate $c$, plus a way to differentiate the error through the solver.
- This generalizes: unknown **diffusion coefficient**, **source terms**, **boundary
  fluxes**, or **spatially varying** parameters all slot in the same way — add a
  trainable parameter (or a small sub-network) and let autograd fit it to the data.

**Try it:** change `C_TRUE`, increase `NOISE`, or reduce `N_DATA` and watch how the
recovered `c` degrades gracefully.